# SupeRR Selector — player ratings

This notebook reads the silver files and builds two lists: batters and bowlers.

Each player has a baseline from their usual rates, then three pressure bands, a within-match comparison, a clutch term, form, and consistency. A career rate is the decay-weighted mean of the rows that player has. A season with no rows is left out of both the total and the weight. The weights are 0.6561, 0.729, 0.81, 0.9, and 1 for 2022 through 2026.

The calculations live in `player_ratings.py`. This notebook runs them and shows the tables. The silver files are not changed.

## Load and tag balls

A legal ball is not a wide, a no-ball, or a super over. A wide or a no-ball adds one conceded run. Bye and leg-bye amounts are not in the file, and neither are extra runs beyond that one-run penalty, so a team total is still a lower bound. An ordinary chase target is the first-innings total plus one. A DLS target is used as supplied.

A legal ball is not a wide, a no-ball, or a super over. Pressure is the occasion times the competition times the ball, plus a shock from the last six balls and a break from the previous eighteen. The current ball is not included. The shock starts with dots and wickets for a batter, and fours and sixes for a bowler, then adds the wickets above xW and the charged runs above xR on those same six balls. Bowling bands are cut inside pace and inside spin, so the new ball does not put more pace bowlers into the high band. A recognized batter is anyone at positions 1 to 7. Zero, medium, and high are the bottom half, the next 30 percent, and the top fifth of that pressure.

In [1]:
from pathlib import Path

import pandas as pd

import player_ratings as ratings

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 30)

balls, matches, comps = ratings.load()
work, legal = ratings.add_situation(balls)
print("deliveries", len(balls), "legal, excluding super overs", len(legal))
print("five-season weight total", round(ratings.W_SUM, 4))
bands = ratings.band_summary(legal)
display(bands.assign(share=lambda frame: (100 * frame["share"]).round(1)))

deliveries 200999 legal, excluding super overs 192675
five-season weight total 4.0951


,balls,share
band,,
zero,94837,49.2
medium,59221,30.7
high,38617,20.0


## Elo

Team strength updates after each match. Batting strength and bowling strength update after each legal ball, on three ladders: IPL, T20I, and lower T20. Emerging Teams and Rising Stars are the lower ladder. A rating learned there does not carry into a T20I, and a ball there counts as half a ball in the career rates and the ranking floor. A player with few balls is held near 1500, and that hold loosens as the career lengthens. Venue strength uses only balls already bowled at that ground. Expected runs and the expected wicket chance are also taken from balls already bowled, split by phase, pace or spin, and ladder, and pulled toward the coarser rate with 30 balls. Before any ball, the starting assumption is 1.0 run and a 0.05 wicket chance. A batter is priced on bat runs. A bowler is priced on charged runs, which are 0 on a bye, leg-bye, or penalty, and the bowling rating uses that same charged total.

In [2]:
charged_gap = legal["bat_runs"] - legal["charged_runs"]
print("balls where bat runs differ from charged runs", int(charged_gap.ne(0).sum()))
print("sum of bat runs minus charged runs", round(float(charged_gap.sum()), 3))
legal = ratings.fit_elo(legal)
summary = legal.attrs["elo_summary"]
for key, value in summary.items():
    if isinstance(value, str):
        print(key + ":", value)
    else:
        print(key + ":", round(value, 1))

balls where bat runs differ from charged runs 1
sum of bat runs minus charged runs 1.0


k_run: 4.0
k_wicket: 16.0
k_team: 20.0
batter_elo_min: 930.6
batter_elo_max: 2573.6
batter_elo_sd: 141.8
bowler_elo_min: 1061.9
bowler_elo_max: 2109.2
bowler_elo_sd: 117.7
team_elo_min: 1361.5
team_elo_max: 1647.1
grounds: 97
step_sizes: as written


## Batter scores

Baseline is strike rate and average against the phases that batter actually plays. The three bands are the strength-adjusted ball value inside zero, medium, and high pressure. Peers compare an innings with teammates and the opposition, and the opposition gap is capped at 3, the same cap as the teammate gap. Clutch uses a high-band spell against the player's own rate. A spell below that rate stays negative. A spell above it is scaled by how far the player also beat their teammates, and a negative teammate gap does not add a bonus. Form uses up to eight earlier innings and is pulled toward the middle when that sample is short. Consistency prefers a narrower spread of match values.

Each term is a z-score inside the batting role, so a top-order batter is compared with other top-order batters, a middle-order batter with the middle order, and a lower-order batter with the lower order. Inside each role the reference is still players with at least 30 decay-weighted legal balls, and each z-score stops at 3. The composite is 0.20 baseline, 0.10 zero, 0.15 medium, 0.20 high, 0.15 clutch, 0.10 peers, 0.05 form, and 0.05 consistency. The extra 0.05, which the eight original shares did not account for, is on the baseline.

In [3]:
batters = ratings.score_batters(legal)
print("batters", len(batters), "ranked", int(batters["ranked"].sum()))
batter_cols = [
    "rank", "player_name", "role", "composite", "baseline",
    "band_zero", "band_medium", "band_high", "clutch", "peers", "form", "consistency",
    "runs", "strike_rate", "average", "n_eff", "high_n_eff",
]
display(ratings.top_table(batters, batter_cols))
print("roles in the batter top 10")
print(batters.loc[batters["rank"].le(10), "role"].value_counts().to_string())

batters 888 ranked 217


,rank,player_name,role,composite,baseline,band_zero,band_medium,band_high,clutch,peers,form,consistency,runs,strike_rate,average,n_eff,high_n_eff
0,1.0,Rilee Rossouw,top order,1.330,1.387,2.082,0.250,2.998,0.005,2.019,-0.599,0.687,717,172.771,32.591,302.519,56.303
1,2.0,Tim David,middle order,1.262,2.374,1.696,1.969,0.455,-0.428,3.000,0.569,-0.674,2153,177.202,34.726,1002.709,314.757
2,3.0,Mahendra Singh Dhoni,lower order,1.223,1.808,-0.848,1.480,1.067,3.000,1.609,0.048,-2.044,690,150.000,31.364,352.416,128.987
3,4.0,George Linde,lower order,1.191,1.089,-0.443,1.981,2.033,0.309,1.742,2.202,-0.344,227,162.143,18.917,130.740,61.550
4,5.0,Ashutosh Sharma,lower order,1.060,2.120,0.474,1.443,1.628,-0.122,0.815,0.175,-0.510,570,167.708,28.175,301.900,93.400
5,6.0,Vaibhav Sooryavanshi,top order,1.058,2.465,3.000,3.000,0.001,-2.202,3.000,-0.096,-3.000,1509,222.320,42.106,608.000,41.650
6,7.0,Tom Banton,middle order,1.005,1.204,1.547,0.680,2.252,-0.071,1.414,-0.723,-0.743,437,159.489,31.214,239.776,68.642
7,8.0,Donovan Ferreira,middle order,0.998,0.952,0.436,2.231,0.968,2.001,1.007,-0.920,-2.376,561,164.035,24.391,317.418,85.793
8,9.0,Dinesh Karthik,middle order,0.996,1.517,1.121,1.903,-0.511,2.050,1.812,0.136,-1.964,1061,166.562,31.206,452.142,152.280
9,10.0,Patrick Cummins,lower order,0.989,1.137,2.164,0.585,0.139,2.876,1.421,-0.049,-2.838,389,158.130,20.474,199.933,92.577


roles in the batter top 10
role
middle order    4
lower order     4
top order       2


## Bowler scores

The same shape. Baseline uses economy and balls per wicket, against the usual rate for that bowler's pace or spin and the phases they bowl. A lower rate is a higher score. The high band, clutch, and peers use the same direction.

In [4]:
bowlers = ratings.score_bowlers(legal, work)
print("bowlers", len(bowlers), "ranked", int(bowlers["ranked"].sum()))
bowler_cols = [
    "rank", "player_name", "role", "composite", "baseline",
    "band_zero", "band_medium", "band_high", "clutch", "peers", "form", "consistency",
    "wickets", "economy", "balls_per_wicket", "n_eff", "high_n_eff",
]
display(ratings.top_table(bowlers, bowler_cols))
print("roles in the bowler top 10")
print(bowlers.loc[bowlers["rank"].le(10), "role"].value_counts().to_string())

bowlers 647 ranked 226


,rank,player_name,role,composite,baseline,band_zero,band_medium,band_high,clutch,peers,form,consistency,wickets,economy,balls_per_wicket,n_eff,high_n_eff
0,1.0,Corbin Bosch,Pace,1.337,1.139,1.451,-0.535,2.623,3.000,0.505,0.292,0.079,33.0,8.332,11.152,346.900,83.700
1,2.0,Matthew Humphreys,Spin,1.270,1.230,1.262,0.184,1.461,2.600,1.498,0.054,0.718,20.0,6.866,15.250,277.028,31.330
2,3.0,Imad Wasim,Spin,1.066,1.483,1.152,0.384,1.464,0.017,2.960,0.610,-0.503,15.0,5.951,16.400,188.568,39.204
3,4.0,Mujeeb Ur Rahman,Spin,0.951,1.314,-0.536,0.546,1.240,1.180,1.787,0.490,0.631,48.0,6.783,17.083,675.993,116.528
4,5.0,Jasprit Bumrah,Pace,0.859,1.290,0.507,1.491,0.312,-0.424,3.000,0.591,-0.043,85.0,6.920,20.106,1455.083,302.538
5,6.0,Dwayne Bravo,Pace,0.851,1.024,-0.109,2.285,1.006,0.747,-0.067,-1.375,1.528,16.0,7.923,13.062,137.125,40.022
6,7.0,Shoriful Islam,Pace,0.837,0.556,-0.910,1.018,1.726,2.430,-0.284,-0.983,0.630,41.0,7.972,19.293,650.221,87.655
7,8.0,Sunil Narine,Spin,0.830,0.451,1.209,-0.624,1.056,0.852,3.000,0.159,1.301,64.0,6.773,23.906,1246.702,255.037
8,9.0,Haider Ali,Spin,0.799,1.207,0.031,0.758,1.867,-0.416,1.870,-1.997,0.850,15.0,6.333,18.000,247.800,58.500
9,10.0,Salman Mirza,Pace,0.793,1.617,1.202,0.648,0.764,-0.313,0.910,1.067,0.034,21.0,6.640,15.619,303.000,41.900


roles in the bowler top 10
role
Pace    5
Spin    5


## Rankings

A player is ranked in a skill with at least 120 decay-weighted legal balls, and at least 30 of those in the high band. Anyone under the line stays in the full table with no rank. The two lists are saved under `ratings/`.

In [5]:
out = Path("ratings")
out.mkdir(exist_ok=True)
batters.to_csv(out / "batters.csv", index=False)
bowlers.to_csv(out / "bowlers.csv", index=False)
print("wrote", out / "batters.csv", "and", out / "bowlers.csv")
print("decision: floor is 120 decay-weighted legal balls and 30 high-band balls")
print("decision: career rate is the decay-weighted mean of rows played; the five weights sum to", round(ratings.W_SUM, 4))
print("decision: elo step sizes", summary["step_sizes"])

wrote ratings\batters.csv and ratings\bowlers.csv
decision: floor is 120 decay-weighted legal balls and 30 high-band balls
decision: career rate is the decay-weighted mean of rows played; the five weights sum to 4.0951
decision: elo step sizes as written


## Deck exports

These files are the evidence behind each stage: cleaning counts, pressure distributions, Elo bounds, calibration, component weights, and the rankings. They are written from the frames above.

In [6]:
notes = ratings.write_deck_exports(balls, work, legal, batters, bowlers)
print("wrote exports/")
for item, value in notes.items():
    print(item + ":", value)

wrote exports/
sample_match_id: 40767
sample_innings: 2
sample_date: 2025-06-03
sample_stage: Final
sample_competition: IPL 2025
sample_home: Royal Challengers Bengaluru
sample_away: Punjab Kings
trajectory_player: Jos Buttler
trajectory_up: 45394
trajectory_down: 20206
trajectory_up_date: 2026-07-11
trajectory_down_date: 2023-04-19
trajectory_up_runs: 131.0
trajectory_down_runs: 36.0
